# 01 - Shared OCR Pipeline Build and Validation
This notebook validates the reusable OCR infrastructure in `src/ocr_engine.py`.

It demonstrates:
- cache-aware OCR execution
- word/line/block structured outputs
- diagnostics image output
- downstream loaders for text, layout, and invoice extraction use-cases

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import load_ocr_config
from src.ocr_engine import (
    check_tesseract_installation,
    ocr_batch,
    ocr_document,
    load_ocr_result,
    load_ocr_text,
    load_ocr_words,
    load_ocr_lines,
    load_ocr_blocks,
)

## 1) Load OCR Configuration

In [2]:
cfg = load_ocr_config(PROJECT_ROOT / 'configs' / 'config.yaml')
# Force cache/output paths to project root (avoid notebooks/data leakage)
cfg.cache_dir = str(PROJECT_ROOT / 'data' / 'interim' / 'ocr')
cfg.diagnostics_dir = str(PROJECT_ROOT / 'outputs' / 'ocr_diagnostics')
cfg

OCRConfig(tesseract_cmd=None, lang='eng', psm=6, oem=3, extra_config='', preprocess_mode='adaptive', enable_grayscale=True, enable_denoise=True, enable_deskew=False, resize_max_dim=1800, min_confidence=0.0, diagnostics_enabled=True, cache_dir='/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/interim/ocr', diagnostics_dir='/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/outputs/ocr_diagnostics', failure_log_path='data/interim/ocr/logs/ocr_failures.jsonl')

## 2) Tesseract Installation Check

In [3]:
tesseract_ok = check_tesseract_installation(cfg.tesseract_cmd, verbose=True)
tesseract_ok

2026-03-30 19:09:34,591 | INFO | ocr_engine | Tesseract available: 5.5.2 | path=/opt/homebrew/bin/tesseract


True

If `False`, install Tesseract and rerun this notebook:\n
- macOS: `brew install tesseract`\n
- Ubuntu/Debian: `sudo apt-get install -y tesseract-ocr`\n
- Windows: install and add to PATH

## 3) Build Full OCR Cache for Train/Val/Test\n
This is the canonical OCR build step for the whole project.\n
It is leakage-safe because OCR is document-level preprocessing and does not use labels for fitting.

In [4]:
train_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'train.csv')
val_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'val.csv')
test_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'test.csv')

for df, split_name in [(train_df, 'train'), (val_df, 'val'), (test_df, 'test')]:
    if 'split' not in df.columns:
        df['split'] = split_name

all_meta = pd.concat([train_df, val_df, test_df], ignore_index=True)
all_meta = all_meta[['doc_id', 'file_path', 'split', 'class_name']].copy()
parsed_dir = Path(cfg.cache_dir) / 'parsed'
parsed_dir.mkdir(parents=True, exist_ok=True)
all_meta['parsed_exists'] = all_meta['doc_id'].map(lambda d: (parsed_dir / f'{d}.json').exists())
missing_ocr = all_meta[~all_meta['parsed_exists']].copy()
print('Total docs:', len(all_meta))
print('Cached docs:', int(all_meta['parsed_exists'].sum()))
print('Missing docs:', len(missing_ocr))
all_meta.head()

Total docs: 12540
Cached docs: 5
Missing docs: 12535


,doc_id,file_path,split,class_name,parsed_exists
0,doc_00000000,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,False
1,doc_00000001,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,False
2,doc_00000002,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,False
3,doc_00000003,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,False
4,doc_00000004,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,False


## 4) Run OCR for Missing Documents + Validation Sample

In [5]:
if not tesseract_ok:
    raise RuntimeError('Tesseract is required for OCR execution.')

RUN_FULL_CACHE_BUILD = True  # set False to skip long full-cache build
if RUN_FULL_CACHE_BUILD and len(missing_ocr) > 0:
    full_build_summary = ocr_batch(missing_ocr[['doc_id','file_path','split','class_name']], cfg=cfg, force=False, save_diagnostics=False, show_progress=True)
    display(full_build_summary.head())
else:
    full_build_summary = Non

# Small benchmark sample (1 per class) to report runtime/quality.
sample = all_meta.groupby('class_name', group_keys=False).head(1).head(5).copy()
benchmark = ocr_batch(sample[['doc_id','file_path','split','class_name']], cfg=cfg, force=False, save_diagnostics=True, show_progress=True)
benchmark

OCR batch: 100%|██████████| 12535/12535 [18:53:14<00:00,  5.42s/it]    


,doc_id,file_path,split,class_name,status,error,runtime_sec,num_words,avg_conf,cache_hit
0,doc_00000000,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,0.669886,160,64.157826,False
1,doc_00000001,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,1.336792,334,25.069066,False
2,doc_00000002,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,0.677828,165,39.087161,False
3,doc_00000003,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,1.346028,361,40.815804,False
4,doc_00000004,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,0.802533,236,54.066119,False


OCR batch: 100%|██████████| 5/5 [00:00<00:00, 52.18it/s]


,doc_id,file_path,split,class_name,status,error,runtime_sec,num_words,avg_conf,cache_hit
0,doc_00000000,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,form,ok,None,0.029162,160,64.157826,True
1,doc_00007501,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,resume,ok,None,0.025688,212,83.444478,True
2,doc_00032499,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,email,ok,None,0.013768,40,71.602243,True
3,doc_00035018,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,invoice,ok,None,0.013060,108,50.496606,True
4,doc_00037492,/Users/sofiiaavetisian/Desktop/UNI/statistical...,train,budget,ok,None,0.013739,91,49.542524,True


## 5) Benchmark Table

In [6]:
benchmark_view = benchmark[['doc_id', 'runtime_sec', 'num_words', 'avg_conf', 'cache_hit', 'status']]
benchmark_view

,doc_id,runtime_sec,num_words,avg_conf,cache_hit,status
0,doc_00000000,0.029162,160,64.157826,True,ok
1,doc_00007501,0.025688,212,83.444478,True,ok
2,doc_00032499,0.013768,40,71.602243,True,ok
3,doc_00035018,0.013060,108,50.496606,True,ok
4,doc_00037492,0.013739,91,49.542524,True,ok


## 6) Inspect Canonical OCR Output

In [7]:
doc_id = sample.iloc[0]['doc_id']
result = load_ocr_result(doc_id, cfg=cfg)
{
    'doc_id': result['doc_id'],
    'num_words': result['stats']['num_words'],
    'num_lines': result['stats']['num_lines'],
    'avg_word_conf': result['stats']['avg_word_conf'],
    'cache_hit': result['cache_hit'],
}

{'doc_id': 'doc_00000000',
 'num_words': 160,
 'num_lines': 28,
 'avg_word_conf': 64.1578255125,
 'cache_hit': False}

## 7) Downstream Consumer Helpers

In [8]:
# A) Text-classification usage
text = load_ocr_text(doc_id, cfg=cfg)
print(text[:800])

POA
oN ZAphS
yKennie wonaray
USA.
PHILIP MORRIS USA
MEDIA DEPARTMENT
120 PARK AVENUE, NEW YORK CITY 10017
PHONE: (917) 663-5000
FAX: (817) 663-5313
Date November 25, 1998
to Mike Peters, GM (Adams Outdoor}
Fax (616) 342-5774
@ com Stave Piskor
# of pages (with cover) YW Qu.
Comments
This tecsimile transmission (andor the documente uocompenying \) they contain confidential
Information batanging to the sander, The Information Ls intonded only tee te use af the eddrecave
rently named ‘above. it you are not the Iniended.reviplant, you are heveby notlied thet “any
Giacoaurey copying, distibuton ‘oF the takcng of any action in vllonce of the contents. ot Ihe
informetioa 1» suicty profibited by low. “tt you have receivad th wansmission in error, plause
immediatly natty us By telephone to arrange 


In [9]:
# B) Layout-feature usage
words_df = load_ocr_words(doc_id, cfg=cfg)
lines_df = load_ocr_lines(doc_id, cfg=cfg)
display(words_df.head())
display(lines_df.head())

,word_id,text,conf,left,top,width,height,right,bottom,page_num,block_num,par_num,line_num,norm_left,norm_top,norm_width,norm_height
0,0,POA,0.000000,295,172,165,65,460,237,1,1,1,1,0.391247,0.172,0.218833,0.065
1,1,oN,32.256161,310,213,53,27,363,240,1,1,1,2,0.411141,0.213,0.070292,0.027
2,2,ZAphS,0.000000,380,206,61,43,441,249,1,1,1,2,0.503979,0.206,0.080902,0.043
3,3,yKennie,0.000000,282,231,87,33,369,264,1,1,1,3,0.374005,0.231,0.115385,0.033
4,4,wonaray,29.457750,377,231,47,33,424,264,1,1,1,3,0.500000,0.231,0.062334,0.033


,line_id,text,left,top,width,height,right,bottom,page_num,block_num,par_num,line_num,word_ids
0,0,POA,295,172,165,65,460,237,1,1,1,1,[0]
1,1,oN ZAphS,310,206,131,43,441,249,1,1,1,2,"[1, 2]"
2,2,yKennie wonaray,282,231,142,33,424,264,1,1,1,3,"[3, 4]"
3,3,USA.,349,265,54,13,403,278,1,1,1,4,[5]
4,4,PHILIP MORRIS USA,291,316,168,13,459,329,1,1,1,5,"[6, 7, 8]"


In [10]:
# C) Invoice-extraction usage
blocks_df = load_ocr_blocks(doc_id, cfg=cfg)
display(blocks_df.head())

,block_id,text,left,top,width,height,right,bottom,page_num,block_num
0,0,POA\noN ZAphS\nyKennie wonaray\nUSA.\nPHILIP M...,37,172,661,744,698,916,1,1


## 8) Diagnostics Visualization

In [11]:
diag_path = PROJECT_ROOT / cfg.diagnostics_dir / f'{doc_id}.png'
if diag_path.exists():
    img = plt.imread(diag_path)
    plt.figure(figsize=(10, 12))
    plt.imshow(img)
    plt.title(f'OCR diagnostics: {doc_id}')
    plt.axis('off')
else:
    print('Diagnostics image not found:', diag_path)

Diagnostics image not found: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/outputs/ocr_diagnostics/doc_00000000.png


## 9) Verify Cache Files

In [12]:
cache_root = PROJECT_ROOT / cfg.cache_dir
print('raw :', cache_root / 'raw' / f'{doc_id}.csv')
print('json:', cache_root / 'parsed' / f'{doc_id}.json')
print('text:', cache_root / 'text' / f'{doc_id}.txt')

raw : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/interim/ocr/raw/doc_00000000.csv
json: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/interim/ocr/parsed/doc_00000000.json
text: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/interim/ocr/text/doc_00000000.txt
